# Clean SEEA 36-account human coding for Sankey diagram

In [2]:
from pathlib import Path
import pandas as pd
import re

# ============================================================
# Stage. Clean SEEA 36-account human coding for Sankey diagram
# ============================================================

# Input file from human coding
INPUT_FILE = Path("SEEA_36_account_LLM_coding_workflow - Human_MC.xlsx")

# Output cleaned file for Sankey diagram
OUTPUT_FILE = Path("SEEA_36_account_final.xlsx")

# Change this if your sheet name is different
INPUT_SHEET = "Step5_Human_Validation"


# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def clean_text(value):
    """Clean text values."""
    if pd.isna(value):
        return ""
    return str(value).strip()


def clean_column_name(col):
    """Standardize column names."""
    col = str(col).strip()
    col = re.sub(r"\s+", " ", col)
    return col


def is_positive_flag(value):
    """
    Convert human/LLM coding values into 1/0.

    Treated as 1:
    1, A, Y, YES, TRUE, APPLICABLE, X

    Treated as 0:
    0, blank, N, NO, FALSE, NOT APPLICABLE, NA
    """
    text = clean_text(value).upper()

    if text in {"1", "A", "Y", "YES", "TRUE", "APPLICABLE", "X"}:
        return 1

    if text in {"", "0", "N", "NO", "FALSE", "NOT APPLICABLE", "N/A", "NA", "NONE"}:
        return 0

    # Try numeric conversion if the value is something like 1.0
    try:
        return 1 if float(text) > 0 else 0
    except ValueError:
        return 0


def classify_dataset_from_source(source):
    """
    Classify indicator source into WB, IMF, or UNSD.
    This follows the same logic used in the Sankey script.
    """
    text = str(source or "").lower()

    if (
        "unstats.un.org/unsd/envstats" in text
        or "united nations comtrade" in text
        or "comtrade.un.org" in text
        or "department of economic and social affairs/united nations" in text
    ):
        return "UNSD"

    if (
        "changing wealth of nations" in text
        or "cwon" in text
        or "worldbank.org" in text
        or "findex" in text
        or "worldwide governance indicators" in text
    ):
        return "WB"

    return "IMF"


# ------------------------------------------------------------
# 2. Load the human-coded workbook
# ------------------------------------------------------------

df = pd.read_excel(INPUT_FILE, sheet_name=INPUT_SHEET)

# Clean column names
df.columns = [clean_column_name(col) for col in df.columns]

print("Rows loaded:", len(df))
print("Columns loaded:", len(df.columns))


# ------------------------------------------------------------
# 3. Define base columns and true SEEA account columns
# ------------------------------------------------------------

base_cols = [
    "no",
    "indicator",
    "Definition",
    "unit",
    "source",
    "original",
    "dataset",
    "selected_account_codes",
    "selected_accounts",
    "selected_thematic_categories",
    "human_review_needed",
    "selected_accounts_json",
    "Justification",
]

# Keep only base columns that exist in the file
base_cols_existing = [col for col in base_cols if col in df.columns]

# True SEEA account columns are account coding columns.
# This avoids accidentally treating no_account, selected_accounts, notes,
# JSON fields, or summary fields as SEEA account flags.
account_cols = [
    col for col in df.columns
    if col.endswith(" accounts")
]

print("True SEEA account columns:", len(account_cols))

if len(account_cols) == 0:
    raise ValueError(
        "No SEEA account columns found. Check whether account columns end with ' accounts'."
    )


# ------------------------------------------------------------
# 4. Clean base fields
# ------------------------------------------------------------

cleaned = df[base_cols_existing].copy()

if "no" in cleaned.columns:
    cleaned["no"] = pd.to_numeric(cleaned["no"], errors="coerce")

# Drop rows without indicator number
cleaned = cleaned.dropna(subset=["no"]).copy()
cleaned["no"] = cleaned["no"].astype(int)

# Clean text fields
for col in cleaned.columns:
    if col != "no":
        cleaned[col] = cleaned[col].apply(clean_text)


# ------------------------------------------------------------
# 5. Clean SEEA account coding columns into 1/0
# ------------------------------------------------------------

for col in account_cols:
    cleaned[col] = df.loc[cleaned.index, col].apply(is_positive_flag).astype(int)


# ------------------------------------------------------------
# 6. Recalculate no_account correctly
# ------------------------------------------------------------

# Count how many SEEA account columns are flagged for each indicator.
cleaned["account_match_count"] = cleaned[account_cols].sum(axis=1)

# If no SEEA account is flagged, mark no_account = 1.
# If at least one SEEA account is flagged, mark no_account = 0.
cleaned["no_account"] = cleaned["account_match_count"].eq(0).astype(int)

# Optional clearer label for review and figures.
cleaned["No direct SEEA account flag"] = cleaned["no_account"]


# ------------------------------------------------------------
# 7. Add dataset group for Sankey source nodes
# ------------------------------------------------------------

if "dataset" not in cleaned.columns:
    raise ValueError("Column 'dataset' is required to create dataset_group.")

# Standardize dataset values
cleaned["dataset_group"] = (
    cleaned["dataset"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Optional: clean common variations
dataset_map = {
    "WB": "WB",
    "WORLD BANK": "WB",
    "WORLDBANK": "WB",
    "IMF": "IMF",
    "INTERNATIONAL MONETARY FUND": "IMF",
    "UNSD": "UNSD",
    "UN": "UNSD",
    "UNITED NATIONS": "UNSD",
    "UNITED NATIONS STATISTICS DIVISION": "UNSD",
}

cleaned["dataset_group"] = cleaned["dataset_group"].replace(dataset_map)

# Check that all dataset_group values are valid
valid_datasets = {"WB", "IMF", "UNSD"}

invalid_datasets = sorted(
    set(cleaned["dataset_group"].dropna().unique()) - valid_datasets
)

if invalid_datasets:
    raise ValueError(
        f"Unexpected dataset values found in column 'dataset': {invalid_datasets}. "
        "Please revise the dataset column so values are WB, IMF, or UNSD."
    )

# ------------------------------------------------------------
# 8. Check duplicates and sort
# ------------------------------------------------------------

if cleaned["no"].duplicated().any():
    duplicated_ids = cleaned.loc[cleaned["no"].duplicated(), "no"].tolist()
    raise ValueError(f"Duplicate indicator numbers found: {duplicated_ids[:20]}")

cleaned = cleaned.sort_values("no").reset_index(drop=True)


# ------------------------------------------------------------
# 9. Save cleaned Sankey input file
# ------------------------------------------------------------

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    cleaned.to_excel(writer, sheet_name="cleaned_sankey_input", index=False)

print("Done.")
print(f"Cleaned Sankey input saved to: {OUTPUT_FILE}")
print("Rows saved:", len(cleaned))
print("SEEA account columns cleaned:", len(account_cols))
print("Indicators with no direct SEEA account flag:", int(cleaned["no_account"].sum()))
print("Indicators with at least one SEEA account:", int((cleaned["account_match_count"] > 0).sum()))

Rows loaded: 566
Columns loaded: 54
True SEEA account columns: 22
Done.
Cleaned Sankey input saved to: SEEA_36_account_final.xlsx
Rows saved: 566
SEEA account columns cleaned: 22
Indicators with no direct SEEA account flag: 218
Indicators with at least one SEEA account: 348
